In [ ]:
# Truncation experiment: fine-tuned DistilBERT with 512-token inputs (the 256-token run is the reference).
# Everything except max_length matches task_a_binary_distilbert_v2.ipynb, so the two runs are comparable.
import sys, json, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          get_linear_schedule_with_warmup, set_seed)
from sklearn.metrics import ConfusionMatrixDisplay

# Find the repository root (works from any folder) and import the shared project code.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/task_a").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from task_a.binary_metrics import headline_metrics
from task_a.leakage import comparison_key
from task_a.tfidf_baseline import load_rows, group_by_split, evaluate, write_run

In [ ]:
# Load this model's settings plus the baseline settings it shares (dataset path, seed), then dataset v2.
config = json.loads((ROOT / "configs/task_a_binary_distilbert_512_v2.json").read_text(encoding="utf-8"))
base = json.loads((ROOT / config["base_config"]).read_text(encoding="utf-8"))
SEED = base["seed"]
set_seed(SEED)  # seeds Python, NumPy and PyTorch
rows, dataset_sha256 = load_rows(ROOT / base["dataset"],
                                 columns=["example_id", "text", "label", "source_dataset", "split", "family_id", "source_binary_label"])
df = pd.DataFrame(rows)
data = group_by_split(rows)  # {"train": [...], "validation": [...], "test_official": [...], "test_jailbreakhub": [...]}

In [ ]:
# Sanity checks: expected splits and labels, near-duplicate "families" kept inside one split,
# no test prompt also in train/validation (case and spacing ignored), and the official test unchanged.
assert set(df["split"]) == {"train", "validation", "test_official", "test_jailbreakhub"}
assert set(df["label"]) == {0, 1}
assert df.groupby("family_id")["split"].nunique().max() == 1, "a near-duplicate family appears in two splits"

training_keys = set(df.loc[df["split"].isin(["train", "validation"]), "text"].map(comparison_key))
test_keys = df.loc[df["split"].isin(["test_official", "test_jailbreakhub"]), "text"].map(comparison_key)
assert not test_keys.isin(training_keys).any(), "a test prompt also appears in train/validation"

raw_test = pd.read_parquet(ROOT / "data/raw/wildguard_test.parquet", columns=["prompt", "adversarial"])
official = df[df["split"] == "test_official"]
assert sorted(zip(official["text"], official["label"])) == sorted(zip(raw_test["prompt"], raw_test["adversarial"].astype(int))), \
    "official test differs from data/raw/wildguard_test.parquet"
print(f"All checks passed for {len(df):,} prompts.")

In [ ]:
# Dataset summary: benign and adversarial prompts per split and source.
summary = pd.crosstab([df["split"], df["source_dataset"]], df["label"]).rename(columns={0: "Benign", 1: "Adversarial"})
summary["Total"] = summary["Benign"] + summary["Adversarial"]
summary.reindex(["train", "validation", "test_official", "test_jailbreakhub"], level=0)

In [ ]:
# Load pretrained DistilBERT with a new classification head: a linear layer that maps the sentence
# representation to two scores (benign, adversarial). Fine-tuning trains the head and the encoder together.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()  # mixed precision: faster, less GPU memory
transformers.logging.set_verbosity_error()  # hide the routine "newly initialised head" warning
tokenizer = AutoTokenizer.from_pretrained(config["model_name"])
model = AutoModelForSequenceClassification.from_pretrained(
    config["model_name"], num_labels=2, id2label={0: "BENIGN", 1: "ADVERSARIAL"}, label2id={"BENIGN": 0, "ADVERSARIAL": 1}).to(device)
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (training will be very slow)")

In [ ]:
# Tokenization turns text into word-piece ids; prompts longer than max_length tokens are truncated.
# Padding to a common length happens later, per batch, so short prompts are not padded to max_length.
MAX_LEN = config["max_length"]
encoded, truncated_share = {}, {}
for split in ["train", "validation", "test_official", "test_jailbreakhub"]:
    tokens = tokenizer([r["text"] for r in data[split]], truncation=True, max_length=MAX_LEN)
    encoded[split] = [{"input_ids": ids, "attention_mask": mask, "labels": r["label"]}
                      for ids, mask, r in zip(tokens["input_ids"], tokens["attention_mask"], data[split])]
    truncated_share[split] = float(np.mean([len(ids) == MAX_LEN for ids in tokens["input_ids"]]))
    print(f"{split:<18} {truncated_share[split]:6.1%} of prompts truncated to {MAX_LEN} tokens")

In [ ]:
# Scoring function used for validation after each epoch and for the final test evaluation.
# It returns P(adversarial) for every prompt (a softmax score, not a calibrated probability).
collate = DataCollatorWithPadding(tokenizer)  # pads each batch to its longest prompt

@torch.no_grad()
def adversarial_scores(examples):
    model.eval()
    scores = []
    for batch in DataLoader(examples, batch_size=config["eval_batch_size"], collate_fn=collate):
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(**batch).logits
        scores += torch.softmax(logits.float(), dim=-1)[:, 1].tolist()
    return scores

In [ ]:
# Training setup: shuffled training batches, AdamW optimizer, and a learning rate that warms up then decays linearly.
# Balanced class weights (as in the TF-IDF models) make mistakes on the rarer class cost more in the loss.
train_loader = DataLoader(encoded["train"], batch_size=config["train_batch_size"], shuffle=True, collate_fn=collate,
                          generator=torch.Generator().manual_seed(SEED))
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
total_steps = len(train_loader) * config["max_epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, int(config["warmup_ratio"] * total_steps), total_steps)
class_counts = np.bincount([r["label"] for r in data["train"]], minlength=2)  # [benign, adversarial]
class_weights = torch.tensor(class_counts.sum() / (2 * class_counts), dtype=torch.float, device=device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
# Fine-tune on train only. After each epoch, score validation and keep the best checkpoint by validation macro-F1;
# stop early when it has not improved for `early_stopping_patience` epochs. The test sets are not used here.
history, best_f1, best_epoch, best_state, epochs_without_improvement = [], -1.0, None, None, 0
for epoch in range(1, config["max_epochs"] + 1):
    model.train()
    start, total_loss = time.time(), 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # guards against unusually large updates
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    val_metrics, _ = evaluate(data["validation"], adversarial_scores(encoded["validation"]), config["threshold"])
    val_f1 = val_metrics["all"]["macro_f1"] or 0.0  # undefined (one class never predicted) counts as 0
    history.append({"epoch": epoch, "train_loss": total_loss / len(train_loader), "validation_macro_f1": val_f1})
    print(f"Epoch {epoch}: train loss {history[-1]['train_loss']:.3f} | validation macro-F1 {val_f1:.3f} | "
          f"{(time.time() - start) / 60:.1f} min")
    if val_f1 > best_f1:
        best_f1, best_epoch, epochs_without_improvement = val_f1, epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}  # copy of the best weights
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= config["early_stopping_patience"]:
            print("Validation macro-F1 stopped improving; stopping early.")
            break

model.load_state_dict(best_state)
print(f"\nSelected checkpoint: epoch {best_epoch} (validation macro-F1 = {best_f1:.3f})")

In [ ]:
# With the selected checkpoint fixed, score validation and then each test set exactly once.
split_names = {"validation": "Validation", "test_official": "Official WildGuardMix test", "test_jailbreakhub": "JailbreakHub test"}
results, predictions = {}, []
for split in split_names:
    scores = adversarial_scores(encoded[split])
    results[split], preds = evaluate(data[split], scores, config["threshold"])
    predictions += [{"example_id": r["example_id"], "split": split, "gold": r["label"], "score": round(float(s), 6), "pred": p}
                    for r, s, p in zip(data[split], scores, preds)]

In [ ]:
# Results: "adversarial" scores treat jailbreaks as the positive class; false-positive rate = benign prompts flagged.
table = pd.DataFrame({name: headline_metrics(results[split]["all"]) for split, name in split_names.items()})
table.style.format("{:.3f}", na_rep="n/a")

In [ ]:
# Confusion matrices: rows are the true labels, columns are the model's predictions.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (split, name) in zip(axes, split_names.items()):
    cm = results[split]["all"]["confusion_matrix"]
    counts = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
    ConfusionMatrixDisplay(counts, display_labels=["Benign", "Adversarial"]).plot(ax=ax, cmap="Blues", values_format=",d", colorbar=False)
    ax.set_title(f"{name} (n = {counts.sum():,})")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
fig.suptitle("Fine-tuned DistilBERT, 512-token inputs")
fig.tight_layout()

In [ ]:
# Overfitting check: the same model scored on its own training data vs unseen validation data, per source.
# A large positive gap means the model fits training prompts much better than new ones.
train_results, _ = evaluate(data["train"], adversarial_scores(encoded["train"]), config["threshold"])
overfit = pd.DataFrame({name: {"Train macro-F1": train_results[source]["macro_f1"],
                                "Validation macro-F1": results["validation"][source]["macro_f1"]}
                        for source, name in [("all", "All"), ("wildguardmix", "WildGuardMix"), ("jailbreakhub", "JailbreakHub")]}).T
overfit["Gap (train - validation)"] = overfit["Train macro-F1"] - overfit["Validation macro-F1"]
overfit.style.format("{:.3f}", na_rep="n/a")

In [ ]:
# Save settings, metrics, predictions, the model and the figure to a new folder (the 256-token run is untouched).
changed_labels = int((df["label"] != df["source_binary_label"]).sum())
run_dir = ROOT / config["output_dir"] / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
record = {"created_utc": datetime.now(timezone.utc).isoformat(), "interface": "notebooks/task_a_binary_distilbert_512_v2.ipynb",
          "config": config, "base_config": base, "seed": SEED, "dataset_sha256": dataset_sha256,
          "score_type": "softmax P(adversarial), not calibrated", "selected_epoch": best_epoch,
          "selected_validation_macro_f1": best_f1, "training_history": history, "truncated_share": truncated_share, "train_metrics": train_results,
          "label_provenance": f"WildGuardMix adversarial flag; JailbreakHub labels with {changed_labels} reviewed "
                              "(not human-verified) corrections and 13 ambiguous rows excluded; other JailbreakHub labels unreviewed.",
          "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__}}
write_run(run_dir, record, results, predictions)
model.save_pretrained(run_dir / "model")  # fine-tuned weights + tokenizer, reloadable with from_pretrained
tokenizer.save_pretrained(run_dir / "model")
fig.savefig(run_dir / "confusion_matrices.png", dpi=150, bbox_inches="tight")
print("Results saved to", run_dir.relative_to(ROOT))

In [ ]:
# Compare with the latest saved 256-token run. The input length is chosen by validation macro-F1 only
# (ties keep the shorter, cheaper input); test scores are shown for both lengths but never used to choose.
runs_256 = sorted((ROOT / "results/task_a_binary_distilbert_v2").glob("run_*/run.json"))
record_256 = json.loads(runs_256[-1].read_text(encoding="utf-8"))
metrics_256 = json.loads((runs_256[-1].parent / "metrics.json").read_text(encoding="utf-8"))
settings_256 = {k: v for k, v in record_256["config"].items() if k not in ("experiment", "note", "max_length", "output_dir")}
settings_512 = {k: v for k, v in config.items() if k not in ("experiment", "note", "max_length", "output_dir")}
assert record_256["config"]["max_length"] == 256 and settings_256 == settings_512, "the runs differ in more than max_length"

comparison = {}
for split, name in split_names.items():
    comparison[(name, "256 tokens")] = headline_metrics(metrics_256[split]["all"])
    comparison[(name, f"{MAX_LEN} tokens")] = headline_metrics(results[split]["all"])
comparison = pd.DataFrame(comparison).T.astype(float)
comparison.to_csv(run_dir / "comparison_256_vs_512.csv")
chosen_length = MAX_LEN if best_f1 > record_256["selected_validation_macro_f1"] else 256
print(f"Compared with 256-token run {runs_256[-1].parent.name}")
comparison.style.format("{:.3f}", na_rep="n/a")

In [ ]:
# Conclusion generated from the results table above ("nan" would mean a score is undefined).
official, jailbreakhub = table["Official WildGuardMix test"], table["JailbreakHub test"]
print(f"Selected checkpoint: epoch {best_epoch} by validation macro-F1 ({best_f1:.3f}).")
print(f"Macro-F1: {official['Macro-F1']:.3f} on the official WildGuardMix test, {jailbreakhub['Macro-F1']:.3f} on the JailbreakHub test "
      f"(gap {official['Macro-F1'] - jailbreakhub['Macro-F1']:+.3f}; positive means the official test scored higher).")
print(f"Adversarial recall: {official['Adversarial recall']:.3f} official, {jailbreakhub['Adversarial recall']:.3f} JailbreakHub. "
      f"False-positive rate: {official['False-positive rate']:.3f} official, {jailbreakhub['False-positive rate']:.3f} JailbreakHub.")
print(f"Overfitting check (train minus validation macro-F1): {overfit.loc['All', 'Gap (train - validation)']:+.3f} overall, "
      f"{overfit.loc['WildGuardMix', 'Gap (train - validation)']:+.3f} WildGuardMix, {overfit.loc['JailbreakHub', 'Gap (train - validation)']:+.3f} JailbreakHub.")
print(f"Validation macro-F1: 256 tokens {record_256['selected_validation_macro_f1']:.3f}, {MAX_LEN} tokens {best_f1:.3f}; "
      f"chosen input length (validation only): {chosen_length} tokens.")
earlier_256 = [json.loads(r.read_text(encoding="utf-8"))["selected_validation_macro_f1"] for r in runs_256]
print(f"Run-to-run variation: {len(earlier_256)} saved 256-token run(s) span validation macro-F1 "
      f"{min(earlier_256):.3f}-{max(earlier_256):.3f}; a difference within that range is not evidence either way.")
print(f"JailbreakHub test prompts reaching the input limit: {record_256['truncated_share']['test_jailbreakhub']:.1%} at 256 tokens, "
      f"{truncated_share['test_jailbreakhub']:.1%} at {MAX_LEN} tokens.")
print(f"Main limitation: labels are source-defined; {changed_labels} JailbreakHub labels come from reviewed corrections "
      "that are not human-verified, and the other JailbreakHub labels are unreviewed and may be noisy.")